In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd

In [ ]:
def solve_with_svd(A, b, tol=1e-12):
    """SVD-решатель, возвращающий все компоненты разложения."""
    U, sigma, Vt = svd(A)
    k = np.sum(sigma > tol)
    
    U1 = U[:, :k]
    V1 = Vt[:k, :].T
    Sigma_inv = np.diag(1.0 / sigma[:k])
    
    x_min_norm = V1 @ Sigma_inv @ U1.T @ b
    V2 = Vt[k:, :].T # Базис ядра
    
    return x_min_norm, V2, sigma, k, U1, V1

def print_svd_stats(A_name, x_exact, x_min_norm, V2, k, U1, V1):
    """Универсальная функция для стандартизированного вывода."""
    print(f"{A_name}")
    print("-" * 50)
    print(f"Эффективный ранг (k):      {k}")
    print(f"Размерность U1:            {U1.shape}")
    print(f"Размерность V1:            {V1.shape}")
    print(f"Размерность ядра (V2):     {V2.shape}")
    
    error_min = np.linalg.norm(x_min_norm - x_exact)
    print(f"Ошибка x_min_norm:         {error_min:.2e}")
    
    # Восстановление решения x_exact = x_min_norm + V2 * c
    if V2.shape[1] > 0:
        # Проецируем разность (x_exact - x_min_norm) на базис ядра
        c = V2.T @ (x_exact - x_min_norm)
        x_recovered = x_min_norm + V2 @ c
        error_rec = np.linalg.norm(x_recovered - x_exact)
        print(f"Ошибка после сдвига по ядру: {error_rec:.2e}\n")
    else:
        print("Матрица полного ранга, ядро тривиально.\n")

def plot_singular_values(sigma, title):
    """Отрисовка убывания сингулярных чисел."""
    plt.figure(figsize=(6, 3))
    plt.plot(sigma, 'o-', markersize=4, label=r'$\sigma_i$')
    plt.yscale('log')
    plt.title(title)
    plt.xlabel('Индекс (i)')
    plt.ylabel('Значение (log)')
    plt.grid(True, which="both", ls="--", alpha=0.6)
    plt.legend()
    plt.show()

In [ ]:
n = 100
x_exact = np.arange(1, n + 1, dtype=float)

### Случай 1: Плохо обусловленная верхнетреугольная матрица

In [ ]:
A_upper = np.zeros((n, n))
for i in range(n):
    A_upper[i, i] = 1
    A_upper[i, i+1:] = -1

b_upper = A_upper @ x_exact
x_min_up, V2_up, sig_up, k_up, U1_up, V1_up = solve_with_svd(A_upper, b_upper)

print_svd_stats("СЛУЧАЙ 1: Верхнетреугольная матрица", x_exact, x_min_up, V2_up, k_up, U1_up, V1_up)
plot_singular_values(sig_up, "Спектр: Верхнетреугольная матрица")

### СЛУЧАЙ 2: Вырожденная трехдиагональная матрица

In [ ]:
A_tridiag = np.zeros((n, n))
np.fill_diagonal(A_tridiag, 2)
np.fill_diagonal(A_tridiag[:, 1:], -1)
np.fill_diagonal(A_tridiag[1:, :], -1)
A_tridiag[0, 1] = -2
A_tridiag[n-1, n-2] = -2

b_tridiag = A_tridiag @ x_exact
x_min_tri, V2_tri, sig_tri, k_tri, U1_tri, V1_tri = solve_with_svd(A_tridiag, b_tridiag)

print_svd_stats("СЛУЧАЙ 2: Трехдиагональная матрица", x_exact, x_min_tri, V2_tri, k_tri, U1_tri, V1_tri)
plot_singular_values(sig_tri, "Спектр: Трехдиагональная матрица")

### СЛУЧАЙ 3: Синус-матрица (Низкоранговая)

In [ ]:
h_x = 1.0 / (n - 1)
h_y = 1.0 / (n - 1)
x_vec = np.arange(n) * h_x
y_vec = np.arange(n) * h_y
A_sin = np.outer(np.sin(np.pi * x_vec), np.sin(np.pi * y_vec))

b_sin = A_sin @ x_exact
x_min_sin, V2_sin, sig_sin, k_sin, U1_sin, V1_sin = solve_with_svd(A_sin, b_sin)

print_svd_stats("СЛУЧАЙ 3: Синус-матрица", x_exact, x_min_sin, V2_sin, k_sin, U1_sin, V1_sin)
plot_singular_values(sig_sin, "Спектр: Синус-матрица")